# 🧪 Lab 01: The Unborn Query

This is the smallest faithful version of the production shape found in `satl_pol_dat_econ`: a wide `master` projection, a wide `detail` projection, a two-column `right_outer` join, another projection, and a filter that keeps rows whose master keys are null.

The historical implementation used this pattern to find new detail records. The later fix rewrote it as `left_anti`. We keep both forms here so the planning work can be compared without changing Spark settings or hiding the original query behind a string. The input files and `.schema` files are the fixtures used by the original local test; the notebook also decodes the bracketed binary keys the same way that test does.

In [1]:
from collections import Counter
from time import perf_counter
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType

spark = (SparkSession.builder
    .appName('OOMem Lab 01 - The Unborn Query')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')

from pathlib import Path

data = Path.cwd() / 'data'
def read_fixture(csv_name):
    csv_path = data / csv_name
    schema = StructType.fromDDL(csv_path.with_suffix('.schema').read_text())
    frame = (spark.read.option('header', True).schema(schema).csv(str(csv_path)))
    decoded = []
    for field in frame.schema.fields:
        value = F.col(field.name)
        if field.dataType.simpleString() == 'binary':
            text = F.decode(value, 'UTF-8')
            is_hex = text.rlike(r'^\[[0-9A-Fa-f ]*\]$')
            hex_value = F.regexp_replace(
                F.substring(text, 2, F.length(text) - 2), ' ', ''
            )
            value = F.when(is_hex, F.unhex(hex_value)).otherwise(value)
        decoded.append(value.alias(field.name))
    return frame.select(*decoded)

master = read_fixture('RDV_LGY_SATL_POL_DAT_ECON.csv')
detail = read_fixture('VW_NPSA_LGY_SATL_POL_DAT_ECON.csv')

master_columns = master.columns
detail_columns = detail.columns
m = master.select(*[F.col(c).alias(f'r_{c}') for c in master_columns])
d = detail.select(*[F.col(c).alias(f'v_{c}') for c in detail_columns])
print('master rows:', master.count(), 'columns:', len(master_columns))
print('detail rows:', detail.count(), 'columns:', len(detail_columns))


master rows: 1002 columns: 50


detail rows: 1000 columns: 51


## The historical shape

This is the operation that appeared in the pre-fix commit (`708f49c`). The join is not the final answer: Catalyst must first carry the two-sided projection through the outer join, then inspect the null-key predicate while considering `EliminateOuterJoin`. That is the path visible in the captured `Thread-2` stack.

In [2]:
right_outer = (
    m.join(
        d,
        (F.col('r_HK_L_POLIZA') == F.col('v_HK_L_POLIZAS'))
        & (F.col('r_HD_L_POLIZA') == F.col('v_HD_L_POLIZA')),
        'right_outer',
    )
    .select(*([f'r_{c}' for c in master_columns] + [f'v_{c}' for c in detail_columns]))
    .filter(
        F.col('r_HK_L_POLIZA').isNull()
        | F.col('r_HD_L_POLIZA').isNull()
    )
)

started = perf_counter()
right_plan = right_outer._jdf.queryExecution().optimizedPlan()
right_seconds = perf_counter() - started

print({
    'shape': 'right_outer + project + null-key filter',
    'optimization_seconds': round(right_seconds, 3),
    'plan_nodes': len(right_plan.treeString().splitlines()),
})
print(right_plan.treeString())

{'shape': 'right_outer + project + null-key filter', 'optimization_seconds': 0.211, 'plan_nodes': 7}
Filter (isnull(r_HK_L_POLIZA#204) OR isnull(r_HD_L_POLIZA#205))
+- Join RightOuter, ((r_HK_L_POLIZA#204 = v_HK_L_POLIZAS#254) AND (r_HD_L_POLIZA#205 = v_HD_L_POLIZA#304))
   :- Filter (isnotnull(r_HK_L_POLIZA#204) AND isnotnull(r_HD_L_POLIZA#205))
   :  +- Project [CASE WHEN RLIKE(static_invoke(StringDecode.decode(HK_L_POLIZA#0, UTF-8, false, false)), ^\[[0-9A-Fa-f ]*\]$) THEN unhex(regexp_replace(substring(static_invoke(StringDecode.decode(HK_L_POLIZA#0, UTF-8, false, false)), 2, (length(static_invoke(StringDecode.decode(HK_L_POLIZA#0, UTF-8, false, false))) - 2)),  , , 1), false) ELSE HK_L_POLIZA#0 END AS r_HK_L_POLIZA#204, CASE WHEN RLIKE(static_invoke(StringDecode.decode(HD_L_POLIZA#1, UTF-8, false, false)), ^\[[0-9A-Fa-f ]*\]$) THEN unhex(regexp_replace(substring(static_invoke(StringDecode.decode(HD_L_POLIZA#1, UTF-8, false, false)), 2, (length(static_invoke(StringDecode.decode(HD_

## The equivalent anti join

The later rewrite expresses the intent directly: keep detail rows for which no master row matches both keys. It removes the outer-join null inference from the optimizer's route.

In [3]:
left_anti = (
    d.join(
        m,
        (F.col('v_HK_L_POLIZAS') == F.col('r_HK_L_POLIZA'))
        & (F.col('v_HD_L_POLIZA') == F.col('r_HD_L_POLIZA')),
        'left_anti',
    )
    .select(
        *[f'v_{c}' for c in detail_columns],
    )
)

started = perf_counter()
anti_plan = left_anti._jdf.queryExecution().optimizedPlan()
anti_seconds = perf_counter() - started

print({
    'shape': 'left_anti',
    'optimization_seconds': round(anti_seconds, 3),
    'plan_nodes': len(anti_plan.treeString().splitlines()),
})
print(anti_plan.treeString())

{'shape': 'left_anti', 'optimization_seconds': 0.05, 'plan_nodes': 7}
Project [HK_L_POLIZAS#152 AS v_HK_L_POLIZAS#254, SW_DELETED#102 AS v_SW_DELETED#255, APLICA#103 AS v_APLICA#256, EPIGNIF4#104 AS v_EPIGNIF4#257, EMCANAL#105 AS v_EMCANAL#258, IDCANAL#106 AS v_IDCANAL#259, TIPPRIM#107 AS v_TIPPRIM#260, INDCART#108 AS v_INDCART#261, SW_OPERACION_VIGENTE#109 AS v_SW_OPERACION_VIGENTE#262, HC_PROVISION_MATEMATICA#110 AS v_HC_PROVISION_MATEMATICA#263, HC_2000_PRIMA_NETA_CONTABLE#111 AS v_HC_2000_PRIMA_NETA_CONTABLE#264, HC_2902_PRIMA_COMERCIAL_INICIO#112 AS v_HC_2902_PRIMA_COMERCIAL_INICIO#265, HC_2903_PRIMA_COMERCIAL_ANUAL#113 AS v_HC_2903_PRIMA_COMERCIAL_ANUAL#266, HC_7100_CAPITAL_ASEG_BASICO#114 AS v_HC_7100_CAPITAL_ASEG_BASICO#267, HC_7200_CAPITAL_RIESGO#115 AS v_HC_7200_CAPITAL_RIESGO#268, HC_7600_CAPITAL_ASEG_BASICO#116 AS v_HC_7600_CAPITAL_ASEG_BASICO#269, HC_7601_CAPITAL_ASEG_COMPLEMEN#117 AS v_HC_7601_CAPITAL_ASEG_COMPLEMEN#270, HC_7610_CAPITAL_ASEG_CONTINENT#118 AS v_HC_7610_CAP

In [4]:
right_rows = right_outer.select('v_HK_L_POLIZAS', 'v_HD_L_POLIZA').collect()
anti_rows = left_anti.select('v_HK_L_POLIZAS', 'v_HD_L_POLIZA').collect()

assert Counter(map(tuple, right_rows)) == Counter(map(tuple, anti_rows))
assert len(right_rows) == len(anti_rows)

print('right_outer rows:', len(right_rows))
print('left_anti rows:', len(anti_rows))
print('PASS: both expressions select the same new detail records.')

right_outer rows: 1000
left_anti rows: 1000
PASS: both expressions select the same new detail records.


## The escalation: the same shape, repeated

The production process did not contain one isolated join. It carried a succession of wide joins and null-key filters through one logical plan. This cell repeats that exact `satl_pol_dat_econ` shape with the real fixture schemas, keeping one input row so the pressure comes from planning, not data volume. Each stage keeps four generated-width projections and retains the optimized plan plus the lazy metadata Catalyst materializes around it (`outputSet`, constraints, expressions, and the canonicalized plan and its metadata). That follows the retained trees in the heap dump and makes the default 1 GB driver cross into a real `OutOfMemoryError`. No Spark memory or optimizer setting is changed. Run this cell in a fresh kernel; after the expected OOM, restart the kernel before running another lab.


In [5]:
from time import perf_counter
from py4j.protocol import Py4JJavaError

STAGES = 20
PROJECTION_COPIES = 4
state = m.limit(1)
detail_one = detail.limit(1)
plan_history = []
metadata_history = []
print('stage | optimize seconds | plan nodes | retained plans | metadata bundles')
print('-' * 64)
try:
    for stage in range(1, STAGES + 1):
        incoming = detail_one.select(
            F.col('HK_L_POLIZAS').alias(f'v{stage}_HK'),
            F.col('HD_L_POLIZA').alias(f'v{stage}_HD'),
            *[
                F.col(c).alias(f'v{stage}_{copy}_{c}')
                for copy in range(PROJECTION_COPIES)
                for c in detail_columns
                if c not in ('HK_L_POLIZAS', 'HD_L_POLIZA')
            ],
        )
        state = (state
            .join(
                incoming,
                (F.col('r_HK_L_POLIZA') == F.col(f'v{stage}_HK'))
                & (F.col('r_HD_L_POLIZA') == F.col(f'v{stage}_HD')),
                'right_outer',
            )
            .select(*state.columns, *incoming.columns)
            .filter(
                F.col('r_HK_L_POLIZA').isNull()
                | F.col('r_HD_L_POLIZA').isNull()
            ))
        started = perf_counter()
        plan = state._jdf.queryExecution().optimizedPlan()
        elapsed = perf_counter() - started
        plan_history.append(plan)
        canonical = plan.canonicalized()
        metadata_history.append((
            plan.outputSet(),
            plan.constraints(),
            plan.expressions(),
            canonical,
            canonical.outputSet(),
            canonical.constraints(),
            canonical.expressions(),
        ))
        if stage % 5 == 0:
            print(f'{stage:5d} | {elapsed:16.2f} | {len(plan.treeString().splitlines()):11d} | {len(plan_history):15d} | {len(metadata_history):17d}')
except Py4JJavaError as exc:
    message = str(exc)
    if 'OutOfMemoryError' in message:
        print('OUTCOME: OutOfMemoryError reached during Catalyst planning.')
        print('The failure happened before any task was submitted.')
    else:
        raise
finally:
    try:
        spark.stop()
    except Exception:
        pass

stage | optimize seconds | plan nodes | retained plans | metadata bundles
----------------------------------------------------------------


    5 |             0.74 |          31 |               5 |                 5


   10 |             2.33 |          56 |              10 |                10


   15 |            12.74 |          81 |              15 |                15


OUTCOME: OutOfMemoryError reached during Catalyst planning.
The failure happened before any task was submitted.


## What this lab proves

This notebook uses the real CSV fixtures from the failing local test and proves two things. First, the historical single join is semantically equivalent to the later anti-join rewrite. Second, the same right-outer shape, when carried through the generated wide projections and the plan metadata that the incident retained, consumes the default 1 GB driver heap during logical planning and reports a real `OutOfMemoryError` before execution. The expensive objects are Catalyst trees and their lazy metadata—not rows waiting on executors.